# On-fault rupture propagation: a SeisSol fault-receiver tutorial

**What you will learn**

The companion notebook looked at shaking *away* from the fault. Here we put virtual sensors
**ON the fault plane** ("pickpoints") and watch the **earthquake rupture run along the fault**.
Three questions, three figures:

1. **How fast does the rupture front travel?**  (Figure 1 - rupture velocity)
2. **Is the rupture sub-shear or supershear?**  (Figure 2 - compared to the shear-wave speed)
3. **How strong is the slip pulse as it propagates?**  (Figure 3 - peak slip rate)

You can compare **one, two, or three (or more) cases** (e.g. different material models) at once.

> All equations are in plain ASCII (no rendered math) on purpose.


## The data: on-fault pickpoints

A **pickpoint** sits directly on the fault. For each one SeisSol writes a time series:
```
columns:  Time, SRs, SRd, T_s, T_d, P_n, Sls, Sld
```
- `SRs, SRd` = **slip rate** (strike, dip) in m/s - how fast the two fault walls slide past each other.
- `T_s, T_d` = shear-traction **change** (zero until the rupture arrives).
- `P_n`      = normal-stress change.
- `Sls, Sld` = **slip** (strike, dip) in m.

The pickpoints we use lie along a line at about **7 km depth**, marching **NW from the
hypocenter** (0, ~10, ~20, ~40 km away). Watching when the **slip rate switches on** at each
one tells us when the rupture front arrived there - and that gives the rupture speed.


## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Imports

In [ ]:
import os, re, glob
import numpy as np
import matplotlib.pyplot as plt

## 3. Point to the cases you want to compare

`RUN_DIRS` maps a **label** to the folder holding that case's `safs-faultreceiver-*.dat`
files. Add or remove entries freely - the notebook plots whichever folders exist, so it
works for one, two, or three (or more) cases.

In [ ]:
# EDIT this dict: label -> Drive folder (must contain safs-faultreceiver-*.dat)
RUN_DIRS = {
    "constant": "/content/drive/MyDrive/seisol_quakeworx/output_safs_v2.2.0_constant_mat_onfaultpoints",
    "CVM":      "/content/drive/MyDrive/seisol_quakeworx/output_safs_v2.2.0_cvm_mat",
    # "case 3":  "/content/drive/MyDrive/seisol_quakeworx/output_safs_v2.2.0_other",
}
for name, d in RUN_DIRS.items():
    n = len(glob.glob(os.path.join(d, "safs-faultreceiver-*.dat"))) if os.path.isdir(d) else 0
    print(f"{name:10s}: {'found' if n else 'MISSING'}  ({n} fault-receiver files)  {d}")

## 4a. The along-strike pickpoints, and the distance along the fault

Below are the pickpoint coordinates (UTM metres; z is negative = depth). The rupture travels
**along the fault surface**, so the distance that matters is the **distance measured along the
fault between consecutive points** - which we compute from the 3-D coordinates (the nominal
"10/20/40 km" labels are only approximate, because each point was snapped to the nearest
fault element at 7 km depth).

In [ ]:
# label -> (x_UTM, y_UTM, z_UTM, nominal along-strike km)
ALONGSTRIKE = {
    "strike_0km":  (608652.1569, 3707403.4118, -6994.8709,  0),
    "strike_10km": (598161.1226, 3717174.8798, -6996.8368, 10),
    "strike_20km": (591590.3682, 3723261.4195, -7012.1792, 20),
    "strike_40km": (576136.0204, 3737161.6404, -6999.2706, 40),
}
HYPOCENTER = (607030.1076, 3707288.7944, -5054.2562)   # for the map only

# distance ALONG the fault = cumulative 3-D distance between consecutive points, in km
order  = sorted(ALONGSTRIKE.items(), key=lambda kv: kv[1][3])     # by nominal distance
xyz    = np.array([v[:3] for _, v in order])
seglen = np.linalg.norm(np.diff(xyz, axis=0), axis=1)
cumkm  = np.concatenate([[0.0], np.cumsum(seglen)]) / 1000.0
DIST   = {lab: cumkm[i] for i, (lab, _) in enumerate(order)}      # label -> along-fault km
print("along-fault distance of each pickpoint:")
for lab, _ in order:
    print(f"  {lab:12s} -> {DIST[lab]:5.1f} km")

## 4b. The shear-wave speed (needed for "supershear")

A rupture is **supershear** when its front outruns the **shear (S) wave** of the rock,
i.e. v_rupture > Cs, where

```
Cs = sqrt( mu / rho )           mu = shear modulus, rho = density
```

For the **constant** material here (`mu = 3.2e10 Pa`, `rho = 2670 kg/m^3`):
`Cs = sqrt(3.2e10 / 2670) = 3460 m/s = 3.46 km/s`. The **Rayleigh speed** `Cr ~ 0.92*Cs`
is the fastest a *sub*-shear rupture can normally go. **Set `CS_KMS` to your model's value.**

In [ ]:
CS_KMS = 3.46              # shear-wave speed [km/s]  (= sqrt(mu/rho); EDIT for your model)
CR_KMS = 0.92 * CS_KMS    # Rayleigh speed [km/s] - the sub-shear speed limit
ONSET_THRESHOLD = 0.05    # slip-rate [m/s] that marks "the rupture has arrived"
print(f"Cs = {CS_KMS:.2f} km/s   Cr = {CR_KMS:.2f} km/s")

## 4c. Map of the pickpoints

Where the sensors are, in map view (relative to the first point, in km). The rupture starts
near the **hypocenter (star)** and runs **NW** through points 1 -> 4. They are all at ~7 km depth.

In [ ]:
p0 = np.array(ALONGSTRIKE["strike_0km"][:2])
station_no = {lab: i+1 for i, (lab, _) in enumerate(order)}

fig, ax = plt.subplots(figsize=(7, 6.5))
pts = {lab: ((v[0]-p0[0])/1000, (v[1]-p0[1])/1000) for lab, v in ALONGSTRIKE.items()}
line = np.array([pts[lab] for lab, _ in order])
ax.plot(line[:,0], line[:,1], "-", color="0.6", lw=2, zorder=1, label="fault (along strike)")
for lab, v in ALONGSTRIKE.items():
    dx, dy = pts[lab]
    ax.scatter(dx, dy, s=110, color="C1", edgecolor="k", zorder=3)
    ax.annotate(f"{station_no[lab]}\n({DIST[lab]:.0f} km)", (dx, dy),
                textcoords="offset points", xytext=(8, 6), fontsize=10, fontweight="bold")
hx, hy = (HYPOCENTER[0]-p0[0])/1000, (HYPOCENTER[1]-p0[1])/1000
ax.scatter(hx, hy, marker="*", s=320, color="red", edgecolor="k", zorder=4, label="hypocenter")
# arrow showing propagation direction (point1 -> point4)
ax.annotate("", xy=line[-1], xytext=line[0],
            arrowprops=dict(arrowstyle="->", color="C0", lw=2, alpha=0.6))
ax.text(*(0.5*(line[0]+line[-1]) + np.array([2, -3])), "rupture\npropagates NW",
        color="C0", fontsize=11, fontweight="bold")
ax.set_aspect("equal")
ax.set_xlabel("East of point 1  [km]"); ax.set_ylabel("North of point 1  [km]")
ax.set_title("On-fault pickpoints along strike (~7 km depth)\nnumbers = point ID (along-fault km)")
ax.grid(True, alpha=0.3); ax.legend(loc="lower left", fontsize=9)
plt.tight_layout(); plt.show()

## 5. Read each pickpoint: when did the rupture arrive, and how strong?

For every pickpoint we:
1. match the file to a point by its coordinates,
2. form the **slip-rate magnitude** `SR = sqrt(SRs^2 + SRd^2)`,
3. find the **rupture onset time** = the first time `SR` exceeds `ONSET_THRESHOLD`,
4. record the **peak slip rate** `max(SR)`.

Each loaded case gets its own colour/marker automatically, so the figures handle any number
of cases.

In [ ]:
def read_coords(path):
    xy = {}
    for line in open(path).readlines()[:8]:
        m = re.match(r"#\s*x([123])\s+([-+0-9.eE]+)", line)
        if m: xy[int(m.group(1))] = float(m.group(2))
    return np.array([xy[1], xy[2]])

def load_alongstrike(run_dir):
    """Return {label: dict(dist, onset, peak_sr)} for the along-strike pickpoints."""
    out = {}
    for f in sorted(glob.glob(os.path.join(run_dir, "safs-faultreceiver-*.dat"))):
        xy = read_coords(f)
        label, dmin = min(((lab, np.hypot(xy[0]-v[0], xy[1]-v[1]))
                           for lab, v in ALONGSTRIKE.items()), key=lambda p: p[1])
        if dmin > 300.0:                       # not one of our along-strike points
            continue
        d = np.loadtxt(f, comments="#", skiprows=2)
        t  = d[:, 0]
        sr = np.hypot(d[:, 1], d[:, 2])        # slip-rate magnitude
        above = np.where(sr > ONSET_THRESHOLD)[0]
        out[label] = dict(dist=DIST[label],
                          onset=float(t[above[0]]) if len(above) else np.nan,
                          peak_sr=float(sr.max()))
    return out

runs = {}
for name, d in RUN_DIRS.items():
    if os.path.isdir(d):
        prof = load_alongstrike(d)
        if prof:
            runs[name] = prof
            print(f"{name}: loaded {len(prof)} along-strike pickpoints")
        else:
            print(f"{name}: folder found but no along-strike pickpoints matched")
    else:
        print(f"{name}: folder missing - skipped")
assert runs, "No cases loaded - check RUN_DIRS."

PALETTE = ["C0", "C3", "C2", "C1", "C4"]; MARKERS = ["o", "s", "^", "D", "v"]
STYLE = {n: dict(color=PALETTE[i % len(PALETTE)], marker=MARKERS[i % len(MARKERS)])
         for i, n in enumerate(runs)}
print("\ncases:", list(runs))

def series(prof, key):
    """(distance, value) sorted by distance, dropping NaNs."""
    pts = sorted((d["dist"], d[key]) for d in prof.values() if np.isfinite(d[key]))
    return np.array([p[0] for p in pts]), np.array([p[1] for p in pts])

## Figure 1 - Rupture velocity: how fast the front travels

We plot the **rupture onset time** against **distance along the fault**. A straight line means
a constant rupture speed; **slope = travel-time per km, so v_rupture = 1 / slope**. We fit a
line through each case and print the average rupture velocity.

Things to notice:
- If the points curve (not a straight line), the rupture is **accelerating or decelerating**.
- A steeper line = slower rupture.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5.5))
for name, prof in runs.items():
    x, y = series(prof, "onset")
    ax.plot(x, y, **STYLE[name], ls="none", ms=9, label=f"{name} (data)")
    if len(x) >= 2:
        slope, intat = np.polyfit(x, y, 1)        # onset[s] = slope*dist[km] + intercept
        vrup = 1.0/slope if slope else np.inf
        xf = np.array([x.min(), x.max()])
        ax.plot(xf, slope*xf + intat, "-", color=STYLE[name]["color"], lw=1.5,
                label=f"{name}: v_rup ~ {vrup:.1f} km/s")
ax.set_xlabel("distance along the fault  [km]")
ax.set_ylabel("rupture onset time  [s]")
ax.set_title("Figure 1 - rupture-front arrival vs distance  (slope^-1 = rupture velocity)")
ax.grid(True, alpha=0.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## Figure 2 - Sub-shear or supershear?

Now the **local** rupture velocity between each pair of neighbouring points
(`v = distance_gap / time_gap`), plotted at the midpoint distance, against two reference speeds:

```
Cs = shear-wave speed       -> rupture is SUPERSHEAR if v_rup > Cs
Cr = 0.92*Cs (Rayleigh)     -> the normal speed limit for a SUB-shear rupture
```

The shaded band above `Cs` is the supershear regime. If every point sits **below** the lines,
the rupture is sub-shear.


In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5.5))
vmax = CS_KMS*1.2
for name, prof in runs.items():
    x, y = series(prof, "onset")        # x = distance, y = onset time
    if len(x) < 2: continue
    mid   = 0.5*(x[1:] + x[:-1])
    vloc  = np.diff(x) / np.diff(y)      # km / s = local rupture velocity
    vmax  = max(vmax, np.nanmax(vloc)*1.1)
    ax.plot(mid, vloc, "-", **STYLE[name], label=name)
# reference speeds
ax.axhspan(CS_KMS, vmax, color="red", alpha=0.08)
ax.axhline(CS_KMS, color="red",  ls="--", lw=1.5, label=f"Cs (shear) = {CS_KMS:.2f} km/s")
ax.axhline(CR_KMS, color="0.4",  ls=":",  lw=1.5, label=f"Cr (Rayleigh) = {CR_KMS:.2f} km/s")
ax.text(ax.get_xlim()[1]*0.98, (CS_KMS+vmax)/2, "supershear", color="red",
        ha="right", va="center", fontsize=11, alpha=0.7)
ax.set_ylim(0, vmax)
ax.set_xlabel("distance along the fault  [km]")
ax.set_ylabel("local rupture velocity  [km/s]")
ax.set_title("Figure 2 - rupture velocity vs the shear-wave speed")
ax.grid(True, alpha=0.3); ax.legend(fontsize=8, loc="lower right")
plt.tight_layout(); plt.show()

## Figure 3 - Peak slip rate along strike

The **peak slip rate** (the maximum sliding speed of the fault) at each point, versus distance.
This measures how *energetic* the slip pulse is as the rupture propagates.

Things to notice:
- A **growing** peak slip rate means the rupture is intensifying as it runs (often as it speeds up).
- Compare cases: a stronger material contrast or different friction can change the pulse strength.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5.5))
for name, prof in runs.items():
    x, y = series(prof, "peak_sr")
    ax.plot(x, y, "-", **STYLE[name], label=name)
ax.set_xlabel("distance along the fault  [km]")
ax.set_ylabel("peak slip rate  [m/s]")
ax.set_title("Figure 3 - peak slip rate along strike")
ax.grid(True, alpha=0.3); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

## What to take away

- **Rupture velocity** (Figure 1): the slope of arrival-time vs distance gives how fast the
  earthquake front runs - here a few km/s. Curvature means it is speeding up or slowing down.
- **Sub- vs supershear** (Figure 2): comparing that speed to the shear-wave speed `Cs` tells you
  the rupture regime. Supershear ruptures (v_rup > Cs) radiate a Mach front and are unusually damaging.
- **Peak slip rate** (Figure 3): how strong the slip pulse is, and whether it grows along strike.

### Try it yourself
- Add a third case to `RUN_DIRS` (cell 3) and re-run - all three figures update automatically.
- Change `ONSET_THRESHOLD` (e.g. 0.01 vs 0.1 m/s): does the rupture velocity change much?
- Set `CS_KMS` to a different model's value: does the rupture cross into the supershear band?
